# Accelerometer feature engineering

Original MSc coursework, lightly prepared for the portfolio. Retained figures are historical submission outputs, not a fresh run. Missing datasets and methodological limitations are described in [README.md](README.md). Text logs and machine-specific metadata were removed; see the root EDITS.md for code fixes.


Run from this directory in order: `01-preprocessing.ipynb`, `02-feature-engineering.ipynb`, `03-eda.ipynb`, then `04-gradient-boosting.ipynb` and `05-random-forest.ipynb`.

In [1]:
import pandas as pd
import numpy as np

signals = pd.read_csv('data/signals.csv')
signals_test = pd.read_csv('data/signals_test.csv')
signals_kaggle = pd.read_csv('data/signals_kaggle.csv')

In [2]:
def signal_mean(signal):
    return np.mean(signal)

def signal_std(signal):
    return np.std(signal)

In [3]:
from scipy.signal import butter, filtfilt
import numpy as np

def high_pass_filter(signal, cutoff=0.3, fs=20, order=5):
    """
    Apply a high-pass filter to the signal.
    
    Parameters:
    - signal: The input signal (1D array).
    - cutoff: The cutoff frequency of the filter in Hz.
    - fs: The sampling frequency in Hz.
    - order: The order of the filter.
    
    Returns:
    - Filtered signal.
    """
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)

# Apply the high-pass filter to each axis of the accelerometer data
for dataset in [signals, signals_test, signals_kaggle]:
    for axis in ['x-axis', 'y-axis', 'z-axis']:
        dataset[f'{axis}_filtered'] = dataset.groupby("user_snippet")[axis].transform(
            lambda x: high_pass_filter(x.values, cutoff=0.3, fs=20, order=5) if len(x) > 18 else x
        )

In [4]:
signals['magnitude'] = np.sqrt(signals['x-axis_filtered']**2 + signals['y-axis_filtered']**2 + signals['z-axis_filtered']**2)
signals_test['magnitude'] = np.sqrt(signals_test['x-axis_filtered']**2 + signals_test['y-axis_filtered']**2 + signals_test['z-axis_filtered']**2)
signals_kaggle['magnitude'] = np.sqrt(signals_kaggle['x-axis_filtered']**2 + signals_kaggle['y-axis_filtered']**2 + signals_kaggle['z-axis_filtered']**2)

In [5]:
import numpy as np
from scipy.stats import entropy
from scipy.signal import find_peaks, welch
from scipy.fftpack import fft
from sklearn.decomposition import PCA
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from itertools import permutations
import pywt

def signal_rms(signal):
    return np.sqrt(np.mean(np.square(signal)))

def autocorr_at_lag(signal, lag):
    """Compute autocorrelation at a specific lag."""
    n = len(signal)
    if lag >= n:
        return np.nan
    y1 = signal[:n - lag]
    y2 = signal[lag:]
    return np.corrcoef(y1, y2)[0, 1]

def signal_autocorrelation(signal, max_lag=3):
    """Compute autocorrelations for lags 1 to max_lag."""
    return [autocorr_at_lag(signal, lag) for lag in range(1, max_lag + 1)]


def spectral_peaks(signal, num_peaks=6):
    freqs = np.fft.rfftfreq(len(signal))
    fft_values = np.abs(np.fft.rfft(signal))
    peaks, _ = find_peaks(fft_values)
    peak_heights = fft_values[peaks]
    sorted_indices = np.argsort(peak_heights)[-num_peaks:]
    return freqs[peaks][sorted_indices], peak_heights[sorted_indices]

def total_power_bands(signal, fs=20):
    f, Pxx = welch(signal, fs=fs)
    bands = [(0.3, 2.5), (2.5, 5.5), (5.5, 10)]
    power = []
    for band in bands:
        mask = (f >= band[0]) & (f <= band[1])
        power.append(np.trapz(Pxx[mask], f[mask]))
    return power

def fft_magnitude_entropy(signal):
    fft_vals = np.abs(fft(signal))[:len(signal)//2]
    fft_mag = fft_vals[:3]  # First 3 FFT magnitudes

    # Normalize FFT values to create a probability distribution
    fft_probs = fft_vals / np.sum(fft_vals)
    fft_ent = entropy(fft_probs)

    return fft_mag, fft_ent

def wavelet_packet_features(signal, wavelet='db1', maxlevel=5):
    wp = pywt.WaveletPacket(data=signal, wavelet=wavelet, mode='symmetric', maxlevel=maxlevel)
    
    # Get all terminal nodes at levels 1 to 5
    nodes = [node.path for node in wp.get_level(maxlevel, 'natural')]
    
    # Collect coefficients from all these nodes
    coeffs = np.concatenate([wp[node].data for node in nodes])
    
    abs_sum = np.sum(np.abs(coeffs))
    energy = np.sum(coeffs ** 2)
    
    # Normalise for entropy calculation
    coeffs_prob = np.abs(coeffs) / np.sum(np.abs(coeffs)) if np.sum(np.abs(coeffs)) != 0 else np.ones_like(coeffs)/len(coeffs)
    coeff_entropy = entropy(coeffs_prob)

    return abs_sum, energy, coeff_entropy


def pca_first_component(x,y,z,mg):
    signals_2d = np.column_stack((x, y, z, mg))
    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(signals_2d)
    return pc1.flatten()


def total_harmonic_distortion(signal):
    N = len(signal)
    fft_vals = np.abs(fft(signal))[:N // 2]
    fft_vals[0] = 0  # remove DC

    fundamental_idx = np.argmax(fft_vals)
    fundamental = fft_vals[fundamental_idx]

    harmonics = np.delete(fft_vals, fundamental_idx)
    harmonic_power = np.sum(harmonics**2)

    return np.sqrt(harmonic_power) / fundamental if fundamental != 0 else 0

def max_difference_dxyz(x, y, z):
    d_xyz = np.sqrt(x**2 + y**2 + z**2)
    return np.max(d_xyz)

def total_magnitude_area(x, y, z):
    magnitude = np.sqrt(x**2 + y**2 + z**2)
    return np.sum(magnitude)

def permutation_entropy(signal, order=3, delay=1):
    """
    Compute the Permutation Entropy (PE) of a 1D signal.

    Parameters:
        signal (np.ndarray): 1D time-series array
        order (int): Embedding dimension (length of sequence, n)
        delay (int): Time delay (τ)

    Returns:
        float: Permutation Entropy (PE)
    """
    n = len(signal)
    if n < order * delay:
        return np.nan

    # Generate all possible permutations of order n
    perms = list(permutations(range(order)))
    perm_indices = {p: i for i, p in enumerate(perms)}
    counts = np.zeros(len(perms)
                      )

    # Extract all subsequences and convert to ordinal patterns
    for i in range(n - delay * (order - 1)):
        sorted_index = tuple(np.argsort(signal[i:i + delay * order:delay]))
        counts[perm_indices[sorted_index]] += 1

    # Normalize to get probabilities
    probs = counts / np.sum(counts)
    probs = probs[probs > 0]  # remove zero entries to avoid log(0)

    # Shannon entropy
    pe = -np.sum(probs * np.log(probs))

    return pe

def largest_lyapunov_exponent(signal, emb_dim=10, tau=1, max_t=100):
    """
    Computes the Largest Lyapunov Exponent (LLE) for a 1D time-series signal.
    Based on the small data quantity method in your reference.

    Parameters:
        signal (np.ndarray): 1D array of time-series data (e.g., acceleration x-component)
        emb_dim (int): Embedding dimension for phase space reconstruction
        tau (int): Time delay (lag) for embedding
        max_t (int): Max number of steps for divergence computation

    Returns:
        float: Estimated Largest Lyapunov Exponent (LLE)
    """
    N = len(signal)
    M = N - (emb_dim - 1) * tau
    if M <= 0:
        return np.nan

    # Phase space reconstruction
    Y = np.array([signal[i:i + emb_dim * tau:tau] for i in range(M)])

    # Find nearest neighbours (excluding self)
    nbrs = NearestNeighbors(n_neighbors=2, algorithm='kd_tree').fit(Y)
    distances, indices = nbrs.kneighbors(Y)
    nearest_indices = indices[:, 1]

    # Compute divergence over time
    divergence = []
    for t in range(1, min(max_t, M)):
        divergences_t = []
        for i in range(M - t):
            j = nearest_indices[i]
            if j + t < M and i + t < M:
                d1 = np.linalg.norm(Y[i] - Y[j])
                d2 = np.linalg.norm(Y[i + t] - Y[j + t])
                if d1 > 0 and d2 > 0:
                    divergences_t.append(np.log2(d2 / d1))
        if divergences_t:
            divergence.append(np.mean(divergences_t))

    if not divergence:
        return np.nan

    # Final LLE estimation: average slope of divergence
    lle = np.mean(divergence)
    return lle

def compute_rqa_features(signal, emb_dim=10, tau=1, epsilon=0.1, lmin=2):
    N = len(signal)
    M = N - (emb_dim - 1) * tau
    if M <= 0:
        return np.nan, np.nan, np.nan, np.nan

    # Phase space reconstruction
    embedded = np.array([signal[i:i + emb_dim * tau:tau] for i in range(M)])
    
    # Compute distance matrix and recurrence plot
    D = pairwise_distances(embedded)
    R = (D <= epsilon).astype(int)

    # Recurrence Rate
    RR = np.sum(R) / (M * M)

    # Find diagonal line lengths
    diag_lines = []
    for k in range(-M + 1, M):
        diag = np.diagonal(R, offset=k)
        count = 0
        for val in diag:
            if val == 1:
                count += 1
            elif count >= lmin:
                diag_lines.append(count)
                count = 0
            else:
                count = 0
        if count >= lmin:
            diag_lines.append(count)

    if len(diag_lines) == 0:
        return RR, 0, 0, 0

    diag_lines = np.array(diag_lines)
    P_L = np.bincount(diag_lines)[lmin:]
    L_vals = np.arange(lmin, lmin + len(P_L))

    if np.sum(P_L) == 0:
        return RR, 0, 0, 0

    # Determinism (DET)
    DET = np.sum(L_vals * P_L) / np.sum(diag_lines)

    # Entropy (ENTR)
    p_l = P_L / np.sum(P_L)
    p_l = p_l[p_l > 0]
    ENTR = -np.sum(p_l * np.log(p_l))

    # Average diagonal line length (L)
    L = np.sum(L_vals * P_L) / np.sum(P_L)

    return RR, DET, ENTR, L

In [6]:
feature_rows = []

for snippet_id, group in signals.groupby("user_snippet"):
    x = group["x-axis_filtered"].values
    y = group["y-axis_filtered"].values
    z = group["z-axis_filtered"].values
    mg = group["magnitude"].values
    x_unfiltered = group["x-axis"].values
    y_unfiltered = group["y-axis"].values
    z_unfiltered = group["z-axis"].values

    row = {
        "user_snippet": snippet_id,

        "x_signal_std": signal_std(x_unfiltered),
        "y_signal_std": signal_std(y_unfiltered),
        "z_signal_std": signal_std(z_unfiltered),
        "mg_signal_std": signal_std(mg),

        "x_signal_mean": signal_mean(x_unfiltered),
        "y_signal_mean": signal_mean(y_unfiltered),
        "z_signal_mean": signal_mean(z_unfiltered),
        "mg_signal_mean": signal_mean(mg),

        "x_rms": signal_rms(x),
        "y_rms": signal_rms(y),
        "z_rms": signal_rms(z),
        "mg_rms": signal_rms(mg),

        "x_autocorr": signal_autocorrelation(x),
        "y_autocorr": signal_autocorrelation(y),
        "z_autocorr": signal_autocorrelation(z),
        "mg_autocorr": signal_autocorrelation(mg),

        "x_spectral_peaks": spectral_peaks(x),
        "y_spectral_peaks": spectral_peaks(y),
        "z_spectral_peaks": spectral_peaks(z),
        "mg_spectral_peaks": spectral_peaks(mg),

        "x_total_power_bands": total_power_bands(x),
        "y_total_power_bands": total_power_bands(y),
        "z_total_power_bands": total_power_bands(z),
        "mg_total_power_bands": total_power_bands(mg),

        "x_fft_magnitude_entropy": fft_magnitude_entropy(x),
        "y_fft_magnitude_entropy": fft_magnitude_entropy(y),
        "z_fft_magnitude_entropy": fft_magnitude_entropy(z),
        "mg_fft_magnitude_entropy": fft_magnitude_entropy(mg),

        "x_wavelet_packet_features": wavelet_packet_features(x),
        "y_wavelet_packet_features": wavelet_packet_features(y),
        "z_wavelet_packet_features": wavelet_packet_features(z),
        "mg_wavelet_packet_features": wavelet_packet_features(mg),

        "pca_first_component": pca_first_component(x,y,z,mg),

        "x_rqa_features": compute_rqa_features(x),
        "x_lypaunov_exponent": largest_lyapunov_exponent(x),
        "x_permutation_entropy": permutation_entropy(x),

        "y_rqa_features": compute_rqa_features(y),
        "y_lypaunov_exponent": largest_lyapunov_exponent(y),
        "y_permutation_entropy": permutation_entropy(y),

        "z_rqa_features": compute_rqa_features(z),
        "z_lypaunov_exponent": largest_lyapunov_exponent(z),
        "z_permutation_entropy": permutation_entropy(z),

        "mg_rqa_features": compute_rqa_features(mg),
        "mg_lypaunov_exponent": largest_lyapunov_exponent(mg),
        "mg_permutation_entropy": permutation_entropy(mg),

        "x_total_harmonic_distortion": total_harmonic_distortion(x),
        "y_total_harmonic_distortion": total_harmonic_distortion(y),
        "z_total_harmonic_distortion": total_harmonic_distortion(z),
        "mg_total_harmonic_distortion": total_harmonic_distortion(mg),

        "max_difference_dxyz": max_difference_dxyz(x, y, z),

        "total_magnitude_area": total_magnitude_area(x, y, z),
    }
    feature_rows.append(row)


feature_rows_test = []

for snippet_id, group in signals_test.groupby("user_snippet"):
    x = group["x-axis_filtered"].values
    y = group["y-axis_filtered"].values
    z = group["z-axis_filtered"].values
    mg = group["magnitude"].values
    x_unfiltered = group["x-axis"].values
    y_unfiltered = group["y-axis"].values
    z_unfiltered = group["z-axis"].values

    row = {
        "user_snippet": snippet_id,

        "x_signal_std": signal_std(x_unfiltered),
        "y_signal_std": signal_std(y_unfiltered),
        "z_signal_std": signal_std(z_unfiltered),
        "mg_signal_std": signal_std(mg),

        "x_signal_mean": signal_mean(x_unfiltered),
        "y_signal_mean": signal_mean(y_unfiltered),
        "z_signal_mean": signal_mean(z_unfiltered),
        "mg_signal_mean": signal_mean(mg),

        "x_rms": signal_rms(x),
        "y_rms": signal_rms(y),
        "z_rms": signal_rms(z),
        "mg_rms": signal_rms(mg),

        "x_autocorr": signal_autocorrelation(x),
        "y_autocorr": signal_autocorrelation(y),
        "z_autocorr": signal_autocorrelation(z),
        "mg_autocorr": signal_autocorrelation(mg),

        "x_spectral_peaks": spectral_peaks(x),
        "y_spectral_peaks": spectral_peaks(y),
        "z_spectral_peaks": spectral_peaks(z),
        "mg_spectral_peaks": spectral_peaks(mg),

        "x_total_power_bands": total_power_bands(x),
        "y_total_power_bands": total_power_bands(y),
        "z_total_power_bands": total_power_bands(z),
        "mg_total_power_bands": total_power_bands(mg),

        "x_fft_magnitude_entropy": fft_magnitude_entropy(x),
        "y_fft_magnitude_entropy": fft_magnitude_entropy(y),
        "z_fft_magnitude_entropy": fft_magnitude_entropy(z),
        "mg_fft_magnitude_entropy": fft_magnitude_entropy(mg),

        "x_wavelet_packet_features": wavelet_packet_features(x),
        "y_wavelet_packet_features": wavelet_packet_features(y),
        "z_wavelet_packet_features": wavelet_packet_features(z),
        "mg_wavelet_packet_features": wavelet_packet_features(mg),

        "pca_first_component": pca_first_component(x,y,z,mg),

        "x_rqa_features": compute_rqa_features(x),
        "x_lypaunov_exponent": largest_lyapunov_exponent(x),
        "x_permutation_entropy": permutation_entropy(x),

        "y_rqa_features": compute_rqa_features(y),
        "y_lypaunov_exponent": largest_lyapunov_exponent(y),
        "y_permutation_entropy": permutation_entropy(y),

        "z_rqa_features": compute_rqa_features(z),
        "z_lypaunov_exponent": largest_lyapunov_exponent(z),
        "z_permutation_entropy": permutation_entropy(z),

        "mg_rqa_features": compute_rqa_features(mg),
        "mg_lypaunov_exponent": largest_lyapunov_exponent(mg),
        "mg_permutation_entropy": permutation_entropy(mg),

        "x_total_harmonic_distortion": total_harmonic_distortion(x),
        "y_total_harmonic_distortion": total_harmonic_distortion(y),
        "z_total_harmonic_distortion": total_harmonic_distortion(z),
        "mg_total_harmonic_distortion": total_harmonic_distortion(mg),

        "max_difference_dxyz": max_difference_dxyz(x, y, z),

        "total_magnitude_area": total_magnitude_area(x, y, z),
    }
    feature_rows_test.append(row)


feature_rows_kaggle = []

for snippet_id, group in signals_kaggle.groupby("user_snippet"):
    x = group["x-axis_filtered"].values
    y = group["y-axis_filtered"].values
    z = group["z-axis_filtered"].values
    mg = group["magnitude"].values
    x_unfiltered = group["x-axis"].values
    y_unfiltered = group["y-axis"].values
    z_unfiltered = group["z-axis"].values

    row = {
        "user_snippet": snippet_id,

        "x_signal_std": signal_std(x_unfiltered),
        "y_signal_std": signal_std(y_unfiltered),
        "z_signal_std": signal_std(z_unfiltered),
        "mg_signal_std": signal_std(mg),

        "x_signal_mean": signal_mean(x_unfiltered),
        "y_signal_mean": signal_mean(y_unfiltered),
        "z_signal_mean": signal_mean(z_unfiltered),
        "mg_signal_mean": signal_mean(mg),

        "x_rms": signal_rms(x),
        "y_rms": signal_rms(y),
        "z_rms": signal_rms(z),
        "mg_rms": signal_rms(mg),

        "x_autocorr": signal_autocorrelation(x),
        "y_autocorr": signal_autocorrelation(y),
        "z_autocorr": signal_autocorrelation(z),
        "mg_autocorr": signal_autocorrelation(mg),

        "x_spectral_peaks": spectral_peaks(x),
        "y_spectral_peaks": spectral_peaks(y),
        "z_spectral_peaks": spectral_peaks(z),
        "mg_spectral_peaks": spectral_peaks(mg),

        "x_total_power_bands": total_power_bands(x),
        "y_total_power_bands": total_power_bands(y),
        "z_total_power_bands": total_power_bands(z),
        "mg_total_power_bands": total_power_bands(mg),

        "x_fft_magnitude_entropy": fft_magnitude_entropy(x),
        "y_fft_magnitude_entropy": fft_magnitude_entropy(y),
        "z_fft_magnitude_entropy": fft_magnitude_entropy(z),
        "mg_fft_magnitude_entropy": fft_magnitude_entropy(mg),

        "x_wavelet_packet_features": wavelet_packet_features(x),
        "y_wavelet_packet_features": wavelet_packet_features(y),
        "z_wavelet_packet_features": wavelet_packet_features(z),
        "mg_wavelet_packet_features": wavelet_packet_features(mg),

        "pca_first_component": pca_first_component(x,y,z,mg),

        "x_rqa_features": compute_rqa_features(x),
        "x_lypaunov_exponent": largest_lyapunov_exponent(x),
        "x_permutation_entropy": permutation_entropy(x),

        "y_rqa_features": compute_rqa_features(y),
        "y_lypaunov_exponent": largest_lyapunov_exponent(y),
        "y_permutation_entropy": permutation_entropy(y),

        "z_rqa_features": compute_rqa_features(z),
        "z_lypaunov_exponent": largest_lyapunov_exponent(z),
        "z_permutation_entropy": permutation_entropy(z),

        "mg_rqa_features": compute_rqa_features(mg),
        "mg_lypaunov_exponent": largest_lyapunov_exponent(mg),
        "mg_permutation_entropy": permutation_entropy(mg),

        "x_total_harmonic_distortion": total_harmonic_distortion(x),
        "y_total_harmonic_distortion": total_harmonic_distortion(y),
        "z_total_harmonic_distortion": total_harmonic_distortion(z),
        "mg_total_harmonic_distortion": total_harmonic_distortion(mg),

        "max_difference_dxyz": max_difference_dxyz(x, y, z),

        "total_magnitude_area": total_magnitude_area(x, y, z),
    }
    feature_rows_kaggle.append(row)

In [84]:
print(len(feature_rows))
print(len(feature_rows_test))
print(len(feature_rows_kaggle))

In [93]:
extra_features_df = pd.DataFrame(feature_rows)
non_numeric_entries = extra_features_df.applymap(lambda x: not np.isscalar(x))
non_numeric_columns = non_numeric_entries.any(axis=0)
non_numeric_columns = non_numeric_columns[non_numeric_columns].index.tolist()

for col in non_numeric_columns:
    expanded_cols = pd.DataFrame(extra_features_df[col].tolist(), columns=[f"{col}_{i}" for i in range(len(extra_features_df[col].iloc[0]))])
    extra_features_df = pd.concat([extra_features_df, expanded_cols], axis=1)
    extra_features_df.drop(columns=[col], inplace=True)

non_float_entries = extra_features_df.applymap(lambda x: not isinstance(x, (float, np.float64)))
non_float_columns = non_float_entries.any(axis=0)
non_float_columns = non_float_columns[non_float_columns].index.tolist()

for col in non_float_columns:
    if isinstance(extra_features_df[col].iloc[0], np.ndarray):
        expanded_cols = pd.DataFrame(extra_features_df[col].tolist(), columns=[f"{col}_{i}" for i in range(extra_features_df[col].iloc[0].shape[0])])
        extra_features_df = pd.concat([extra_features_df, expanded_cols], axis=1)
        extra_features_df.drop(columns=[col], inplace=True)

# Merge with original metadata
metadata = pd.read_csv('data/metadata.csv')
extra_features_df = extra_features_df.merge(metadata[["user_snippet", "activity"]], on="user_snippet", how="left")

# Drop user_snippet for training
X_train = extra_features_df.drop(columns=["user_snippet", "activity"])
y_train = extra_features_df["activity"]




extra_features_test_df = pd.DataFrame(feature_rows_test)
non_numeric_entries_test = extra_features_test_df.applymap(lambda x: not np.isscalar(x))
non_numeric_columns_test = non_numeric_entries_test.any(axis=0)
non_numeric_columns_test = non_numeric_columns_test[non_numeric_columns_test].index.tolist()

for col in non_numeric_columns_test:
    expanded_cols = pd.DataFrame(extra_features_test_df[col].tolist(), columns=[f"{col}_{i}" for i in range(len(extra_features_test_df[col].iloc[0]))])
    extra_features_test_df = pd.concat([extra_features_test_df, expanded_cols], axis=1)
    extra_features_test_df.drop(columns=[col], inplace=True)

non_float_entries_test = extra_features_test_df.applymap(lambda x: not isinstance(x, (float, np.float64)))
non_float_columns_test = non_float_entries_test.any(axis=0)
non_float_columns_test = non_float_columns_test[non_float_columns_test].index.tolist()

for col in non_float_columns_test:
    if isinstance(extra_features_test_df[col].iloc[0], np.ndarray):
        expanded_cols = pd.DataFrame(extra_features_test_df[col].tolist(), columns=[f"{col}_{i}" for i in range(extra_features_test_df[col].iloc[0].shape[0])])
        extra_features_test_df = pd.concat([extra_features_test_df, expanded_cols], axis=1)
        extra_features_test_df.drop(columns=[col], inplace=True)

# Merge with original metadata
metadata_test = pd.read_csv('data/metadata_test.csv')
extra_features_test_df = extra_features_test_df.merge(metadata_test[["user_snippet", "activity"]], on="user_snippet", how="left")

# Drop user_snippet for training
X_test = extra_features_test_df.drop(columns=["user_snippet", "activity"])
y_test = extra_features_test_df["activity"]




extra_features_kaggle_df = pd.DataFrame(feature_rows_kaggle)
non_numeric_entries_kaggle = extra_features_kaggle_df.applymap(lambda x: not np.isscalar(x))
non_numeric_columns_kaggle = non_numeric_entries_kaggle.any(axis=0)
non_numeric_columns_kaggle = non_numeric_columns_kaggle[non_numeric_columns_kaggle].index.tolist()

for col in non_numeric_columns_kaggle:
    expanded_cols = pd.DataFrame(extra_features_kaggle_df[col].tolist(), columns=[f"{col}_{i}" for i in range(len(extra_features_kaggle_df[col].iloc[0]))])
    extra_features_kaggle_df = pd.concat([extra_features_kaggle_df, expanded_cols], axis=1)
    extra_features_kaggle_df.drop(columns=[col], inplace=True)

non_float_entries_kaggle = extra_features_kaggle_df.applymap(lambda x: not isinstance(x, (float, np.float64)))
non_float_columns_kaggle = non_float_entries_kaggle.any(axis=0)
non_float_columns_kaggle = non_float_columns_kaggle[non_float_columns_kaggle].index.tolist()

for col in non_float_columns_kaggle:
    if isinstance(extra_features_kaggle_df[col].iloc[0], np.ndarray):
        expanded_cols = pd.DataFrame(extra_features_kaggle_df[col].tolist(), columns=[f"{col}_{i}" for i in range(extra_features_kaggle_df[col].iloc[0].shape[0])])
        extra_features_kaggle_df = pd.concat([extra_features_kaggle_df, expanded_cols], axis=1)
        extra_features_kaggle_df.drop(columns=[col], inplace=True)

# Merge with original metadata
metadata_kaggle = pd.read_csv('data/metadata_kaggle.csv')
extra_features_kaggle_df = extra_features_kaggle_df.merge(metadata_kaggle[["user_snippet"]], on="user_snippet", how="left")

# Drop user_snippet for training
X_kaggle = extra_features_kaggle_df.drop(columns=["user_snippet"])
user_snippet_kaggle = extra_features_kaggle_df["user_snippet"]

In [94]:
print (X_train.shape)
print (X_test.shape)
print (y_train.shape)
print (y_test.shape)
print (X_kaggle.shape)
print (user_snippet_kaggle.shape)

In [ ]:
import os

if not os.path.exists('data-preprocess/new'):
    os.makedirs('data-preprocess/new')

X_train.to_csv('data-preprocess/new/X_train.csv', index=False)
y_train.to_csv('data-preprocess/new/y_train.csv', index=False)
X_test.to_csv('data-preprocess/new/X_test.csv', index=False)
y_test.to_csv('data-preprocess/new/y_test.csv', index=False)
X_kaggle.to_csv('data-preprocess/new/X_kaggle.csv', index=False)
user_snippet_kaggle.to_csv('data-preprocess/new/user_snippet_kaggle.csv', index=False)